# Séance 8 — Génération automatique de visualisations
### Cours "Visualisation de Données"

---

> **🎯 Objectifs de la séance**
> - Comprendre chaque étape du pipeline data→viz piloté par LLM
> - Implémenter un repair loop étape par étape
> - Construire une interface Streamlit minimale puis complète
> - Assembler les micro-étapes en un pipeline autonome

---

## Prérequis techniques

```bash
# Installation des librairies Python
pip install ollama pandas plotly streamlit

# Ollama doit être installé et lancé localement
# Téléchargement : https://ollama.com
# Lancement du service : ollama serve
# Téléchargement du modèle : ollama pull llama3.2
```

**Dataset utilisé :** `job_postings.csv` — 25 114 offres d'emploi, 17 colonnes.

**💡 Conventions :**
- `# NE PAS MODIFIER` — exécutez et lisez attentivement
- `# ✏️ À COMPLÉTER` — complétez les `# TODO`
- Les graphiques s'ouvrent dans votre **navigateur** (pas dans Jupyter)

In [ ]:
# NE PAS MODIFIER - Vérification de l'environnement
import subprocess, sys

libs = ['ollama', 'pandas', 'plotly', 'streamlit']
for lib in libs:
    try:
        __import__(lib)
        print(f"✓ {lib}")
    except ImportError:
        print(f"✗ {lib} manquant — exécutez : pip install {lib}")

import ollama
try:
    models = [m.model for m in ollama.list().models]
    print(f"\n✓ Ollama actif — modèles disponibles : {models}")
    if not any('llama3.2' in m for m in models):
        print("  ⚠️  llama3.2 absent — exécutez : ollama pull llama3.2")
except Exception as e:
    print(f"\n✗ Ollama non disponible : {e}")

---
# PARTIE 1 — COURS THÉORIQUE
---

## 1.1 — Architecture d'un pipeline data→viz

### Le problème
En séance 7, la chaîne était **manuelle** :
1. Écrire le prompt à la main
2. Copier-coller le code généré
3. Corriger les erreurs à la main
4. Exécuter

Un **pipeline** automatise cette chaîne entière : une question → un graphique.

### Architecture en 5 étapes

```
┌─────────────┐    ┌──────────────────┐    ┌─────────────────┐
│  [1] CSV    │───▶│  [2] Résumé      │───▶│  [3] LLM        │
│  DataFrame  │    │  stats + top-N   │    │  → code Plotly  │
└─────────────┘    └──────────────────┘    └────────┬────────┘
                                                    │
                   ┌──────────────────┐    ┌────────▼────────┐
                   │  [5] Rendu HTML  │◀───│  [4] Repair     │
                   │  output.html     │    │  loop (0-3x)    │
                   └──────────────────┘    └─────────────────┘
```

### Pourquoi l'étape 2 est critique
Le LLM ne peut pas lire 25 000 lignes. On lui envoie une **représentation compacte** :
- Statistiques descriptives (`df.describe()`)
- Top valeurs par colonne catégorielle
- Types de colonnes

> La qualité du résumé détermine la qualité du graphique final.

### Sandboxing — exécuter du code généré en sécurité

```python
# ✓ Contexte minimal et sûr — le LLM ne peut pas toucher le système de fichiers
exec(code, {'df': df, 'px': px, 'pd': pd})

# ✗ Dangereux — donne accès à os, sys, open...
exec(code)
```

## 1.2 — Le repair loop

### Pourquoi le code LLM échoue (~40% au premier essai)

| Erreur fréquente | Exemple | Cause |
|-----------------|---------|-------|
| Mauvais paramètre | `category_order=` | LLM confond l'API Plotly |
| Colonne hallucinée | `"Job_Level"` | LLM invente un nom |
| Import manquant | `px.bar(...)` sans import | Contexte tronqué |
| Logique pandas fausse | Mauvais `groupby` | LLM approxime |

### Pattern repair loop

```
Tentative 1 ──▶ exec(code) ──▶ ✓ Succès → afficher le graphique
                    │
                    ✗ Erreur
                    │
Tentative 2 ──▶ prompt + erreur_1 ──▶ LLM ──▶ exec(code_2) ──▶ ✓ / ✗
                                                                    │
                                                                    ✗
Tentative 3 ──▶ historique complet ──▶ LLM ──▶ exec(code_3)
```

### Ce qu'on injecte dans le prompt de réparation

```python
messages.append({'role': 'user', 'content':
    f"Erreur Python : {e}\n"
    f"Colonnes disponibles : {list(df.columns)}\n"
    f"Retourne UNIQUEMENT le code corrigé dans ```python...```"})
```

> **3 tentatives max** — au-delà, le prompt initial est probablement mal formulé.

## 1.3 — Approche classique vs LLM-généré

| Critère | Tableau Public | Pipeline LLM |
|---------|---------------|--------------|
| Temps pour 1 viz | 5–15 min | 10–30 sec |
| Temps pour 100 viz | 500+ min | ~30 min |
| Contrôle visuel | Total | Partiel |
| Reproductibilité | Manuelle | Automatique |
| Nouveau dataset | Refaire manuellement | Relancer le pipeline |
| Fiabilité | 100% | 70–90% |

### La règle de décision

```
Viz unique + présentation soignée  → Tableau Public
Même viz sur N datasets différents → Pipeline LLM
Exploration rapide d'un dataset    → Pipeline LLM
Dashboard interactif               → Streamlit (Exercice 3)
```

---
# PARTIE 2 — TRAVAUX PRATIQUES
---

---
## Exercice 1 — Pipeline étape par étape

> **🎯 Objectif :** Construire le pipeline en 6 micro-étapes indépendantes, chaque cellule ne faisant qu'une seule chose.

> **⏱️ Durée estimée : 55 minutes**

Chaque étape a une cellule démo (`# NE PAS MODIFIER`) suivie d'une cellule à compléter (`# ✏️ À COMPLÉTER`). Exécutez la démo, lisez le code, puis complétez la cellule étudiante.

### Étape 1A — Charger le dataset

Première étape du pipeline : lire le CSV et afficher ses dimensions pour vérifier le chargement.

In [ ]:
# NE PAS MODIFIER - Étape 1A : chargement du dataset
import pandas as pd

df = pd.read_csv('job_postings.csv')
print(f"Shape : {df.shape}")
print(f"Colonnes : {list(df.columns)}")

In [ ]:
# ✏️ À COMPLÉTER — Étape 1A
import pandas as pd

# TODO : Chargez job_postings.csv dans un DataFrame
# TODO : Affichez les 3 premières lignes avec df.head(3)
# TODO : Affichez le nombre de valeurs manquantes par colonne avec df.isna().sum()

# Votre code ici :

### Étape 1B — Résumer le dataset pour le LLM

Le LLM ne peut pas lire 25 000 lignes. On calcule les stats numériques et les top valeurs catégorielles.

In [ ]:
# NE PAS MODIFIER - Étape 1B : stats numériques
import pandas as pd

df = pd.read_csv('job_postings.csv')

# Calcul des stats pour chaque colonne numérique
numeric_stats = {}
for col in df.select_dtypes(include='number').columns:
    s = df[col].dropna()
    if len(s) > 0:
        # On arrondit pour ne pas encombrer le prompt avec trop de décimales
        numeric_stats[col] = {
            'min': round(float(s.min()), 2),
            'max': round(float(s.max()), 2),
            'mean': round(float(s.mean()), 2),
        }

print("Stats numériques :")
for col, stats in numeric_stats.items():
    print(f"  {col}: {stats}")

In [ ]:
# ✏️ À COMPLÉTER — Étape 1B
import pandas as pd

df = pd.read_csv('job_postings.csv')

# Les stats numériques sont calculées dans la démo ci-dessus.
# TODO : Calculez les top-5 valeurs pour chaque colonne catégorielle.
#         Identifiez les colonnes catégorielles avec :
#           [c for c in df.columns if df[c].dtype == object]
#         Puis pour chaque colonne :
#           df[col].value_counts().head(5).index.tolist()

categorical_tops = {}
for col in [c for c in df.columns if df[c].dtype == object]:
    # TODO : remplissez categorical_tops[col] avec les top-5 valeurs
    pass

# TODO : Affichez categorical_tops pour vérifier

### Étape 1C — Construire le prompt

On assemble les infos du dataset dans un f-string structuré. La qualité du prompt détermine la qualité du code généré.

In [ ]:
# NE PAS MODIFIER - Étape 1C : construction du prompt
import pandas as pd

df = pd.read_csv('job_postings.csv')

# Résumé compact : seulement les colonnes et quelques stats clés
colonnes = list(df.columns)
top_industries = df['Company Industry'].value_counts().head(5).index.tolist()

question = "Top 10 des industries avec le plus d'offres, bar chart horizontal"

# Le prompt indique les noms de colonnes EXACTS pour éviter que le LLM les hallucine
prompt = f"""Tu es un expert Python et Plotly. Génère du code de visualisation.

Dataset : job_postings.csv ({df.shape[0]} lignes x {df.shape[1]} colonnes)
Colonnes exactes : {', '.join(colonnes)}
Top industries : {top_industries}

Question : {question}

Contraintes :
- DataFrame déjà chargé dans 'df', imports pandas (pd) et plotly.express (px) disponibles
- Termine avec fig.write_html('output.html')
- Réponds UNIQUEMENT avec le code dans ```python...```
"""

# On affiche les 300 premiers caractères — le prompt complet peut être long
print(f"Prompt ({len(prompt)} caractères) :")
print(prompt[:300], "...")

In [ ]:
# ✏️ À COMPLÉTER — Étape 1C
import pandas as pd

df = pd.read_csv('job_postings.csv')

# TODO : Construisez un prompt pour une AUTRE question analytique
# TODO : Ajoutez dans le prompt les stats numériques de 'Years of Experience' (min/max)
# TODO : Affichez la longueur du prompt avec len(prompt)

ma_question = "Distribution des années d'expérience par niveau de poste, box plot"

# Votre prompt ici :
# prompt = f'''Tu es un expert...'''

### Étape 1D — Appeler le LLM

On envoie le prompt à Ollama et on récupère la réponse brute.

In [ ]:
# NE PAS MODIFIER - Étape 1D : appel LLM
import ollama

# On suppose que 'prompt' a été défini dans la cellule 1C
# Relancez la cellule 1C si besoin avant celle-ci
response = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': prompt}]
)
raw = response.message.content  # API Ollama : response.message.content (pas response['message']['content'])
print(raw[:500])

In [ ]:
# ✏️ À COMPLÉTER — Étape 1D
import ollama

# TODO : Utilisez le prompt que vous avez construit à l'étape 1C (ma_question)
# TODO : Appelez ollama.chat avec model='llama3.2'
# TODO : Affichez les 300 premiers caractères de la réponse

# Votre code ici :

### Étape 1E — Extraire le code Python

Le LLM répond souvent avec du texte explicatif + le code entre balises ` ```python ... ``` `. On extrait uniquement le code.

In [ ]:
# NE PAS MODIFIER - Étape 1E : extraction du code Python
import re

def extract_python_code(text: str) -> str:
    # Pattern 1 : LLM a écrit ```python — cas le plus courant
    match = re.search(r'```python\n(.*?)```', text, re.DOTALL)
    if match:
        return match.group(1).strip()
    # Pattern 2 : LLM a omis le mot "python" dans la balise
    match = re.search(r'```\n(.*?)```', text, re.DOTALL)
    if match:
        return match.group(1).strip()
    # Fallback : on retourne le texte brut si aucune balise trouvée
    return text.strip()

# Test avec la réponse brute de l'étape 1D
code_extrait = extract_python_code(raw)
print("Code extrait :")
print("-" * 40)
print(code_extrait[:400])

In [ ]:
# ✏️ À COMPLÉTER — Étape 1E
import re

# TODO : Testez extract_python_code avec une réponse LLM MALFORMÉE
# (sans balises ```python```) pour voir ce que la fonction retourne

texte_malformed = "Voici le code : import plotly.express as px\nfig = px.bar(df)"
# TODO : appelez extract_python_code(texte_malformed) et affichez le résultat

# TODO : Que se passe-t-il si le LLM répond sans balises de code ?
# Réponse : ...

### Étape 1F — Exécuter le code

On exécute le code dans un namespace restreint (sandboxing) et on ouvre le graphique dans le navigateur.

In [ ]:
# NE PAS MODIFIER - Étape 1F : exécution du code généré
import plotly.express as px, plotly.io as pio, webbrowser

pio.renderers.default = 'browser'

# Namespace minimal : le code LLM n'a accès qu'à df, px et pd — pas à os/sys
context = {'df': df, 'px': px, 'pd': pd}

# On remplace fig.show() par write_html pour éviter le renderer Jupyter
code_a_exec = code_extrait.replace(
    'fig.show()',
    "fig.write_html('output.html')"
)

exec(code_a_exec, context)
webbrowser.open('output.html')
print("✓ Graphique généré -> output.html")

In [ ]:
# ✏️ À COMPLÉTER — Étape 1F
import plotly.express as px, plotly.io as pio, webbrowser

pio.renderers.default = 'browser'

# TODO : Ajoutez une gestion d'erreur avec try/except autour de exec()
# TODO : En cas d'erreur, affichez le message d'erreur et le code fautif

context = {'df': df, 'px': px, 'pd': pd}

# Votre code avec try/except ici :
# try:
#     exec(code_extrait, context)
#     webbrowser.open('output.html')
# except Exception as e:
#     print(f"Erreur : {e}")
#     print("Code fautif :")
#     print(code_extrait)

---
## Exercice 2 — Repair loop

> **🎯 Objectif :** Transformer la gestion d'erreur manuelle de l'Exercice 1 en une boucle automatique qui se répare seule.

> **⏱️ Durée estimée : 35 minutes**

### Étape 2A — Première tentative + capture d'erreur

Une seule itération : appel LLM → exec → capture de l'erreur si elle existe.

In [ ]:
# NE PAS MODIFIER - Étape 2A : une tentative avec capture d'erreur
import ollama, re, pandas as pd, plotly.express as px

df = pd.read_csv('job_postings.csv')

prompt_initial = (
    "Génère du code Plotly Express pour un bar chart horizontal "
    "des 10 industries les plus fréquentes dans df.\n"
    f"Colonnes : {list(df.columns)}\n"
    "DataFrame dans 'df', px disponible. Termine avec fig.write_html('output.html').\n"
    "Réponds uniquement dans ```python...```"
)

# Appel LLM (tentative 1)
response = ollama.chat(model='llama3.2',
                       messages=[{'role': 'user', 'content': prompt_initial}])
contenu = response.message.content

# Extraction du code
match = re.search(r'```python\n(.*?)```', contenu, re.DOTALL)
code_t1 = match.group(1).strip() if match else contenu.strip()

# Exécution avec capture d'erreur
try:
    exec(code_t1, {'df': df, 'px': px, 'pd': pd})
    print("✓ Succès à la tentative 1")
    erreur_t1 = None
except Exception as e:
    erreur_t1 = str(e)
    print(f"✗ Erreur : {erreur_t1}")

In [ ]:
# ✏️ À COMPLÉTER — Étape 2A
import ollama, re, pandas as pd, plotly.express as px

df = pd.read_csv('job_postings.csv')

# TODO : Modifiez le prompt pour demander un box plot de 'Years of Experience'
#         par 'Job Position Level'
# TODO : Exécutez la tentative 1 et capturez l'erreur éventuelle dans erreur_t1
# TODO : Affichez "Succès" ou le message d'erreur

# Votre code ici :

### Étape 2B — Injecter l'erreur dans l'historique

Si la tentative 1 a échoué, on ajoute l'erreur dans l'historique de messages pour que le LLM se corrige.

In [ ]:
# NE PAS MODIFIER - Étape 2B : injection de l'erreur dans l'historique
# Suppose que erreur_t1 et contenu sont définis par la cellule 2A

if erreur_t1:
    # L'historique multi-tour : le LLM voit la conversation complète et comprend son erreur
    messages = [
        {'role': 'user',      'content': prompt_initial},
        {'role': 'assistant', 'content': contenu},          # réponse LLM tentative 1
        {'role': 'user',      'content':
            f"Erreur Python obtenue : {erreur_t1}\n"
            f"Colonnes exactes du DataFrame : {list(df.columns)}\n"
            "Corrige le code. Retourne UNIQUEMENT le code corrigé dans ```python...```"}
    ]
    print(f"Historique construit : {len(messages)} messages")
    print(f"Dernier message : {messages[-1]['content'][:150]}...")
else:
    print("Pas d'erreur — pas besoin de réparation")
    messages = None

In [ ]:
# ✏️ À COMPLÉTER — Étape 2B
# TODO : Si erreur_t1 n'est pas None, construisez l'historique messages (comme dans la démo)
# TODO : Ajoutez dans le message de réparation le min/max de 'Years of Experience'
# TODO : Affichez le nombre de tokens estimé :
#         len(' '.join([m['content'] for m in messages]).split())

# Votre code ici :

### Étape 2C — La boucle complète

On assemble les étapes 2A et 2B dans une fonction qui itère jusqu'au succès ou `max_retries`.

In [ ]:
# NE PAS MODIFIER - Étape 2C : fonction repair_loop complète
import ollama, re, pandas as pd, plotly.express as px, webbrowser

df = pd.read_csv('job_postings.csv')

def repair_loop(prompt: str, df, max_retries: int = 3) -> bool:
    # Namespace restreint : pas d'accès à os/sys depuis le code généré
    context = {'df': df, 'px': px, 'pd': pd}
    messages = [{'role': 'user', 'content': prompt}]

    for attempt in range(1, max_retries + 1):
        print(f"  Tentative {attempt}/{max_retries}...")
        response = ollama.chat(model='llama3.2', messages=messages)
        contenu = response.message.content

        # Extraction du bloc ```python...```
        match = re.search(r'```python\n(.*?)```', contenu, re.DOTALL)
        code = match.group(1).strip() if match else contenu.strip()
        # Remplacement de fig.show() pour éviter l'ouverture dans Jupyter
        code = code.replace('fig.show()', "fig.write_html('output.html')")

        try:
            exec(code, context)
            print(f"  ✓ Succès à la tentative {attempt}")
            webbrowser.open('output.html')
            return True
        except Exception as e:
            print(f"  ✗ Erreur : {e}")
            if attempt < max_retries:
                # On empile : réponse LLM précédente + erreur pour que le LLM comprenne son erreur
                messages.append({'role': 'assistant', 'content': contenu})
                messages.append({'role': 'user', 'content':
                    f"Erreur : {e}\n"
                    f"Colonnes : {list(df.columns)}\n"
                    "Retourne UNIQUEMENT le code corrigé dans ```python...```"})

    print(f"  ✗ Échec après {max_retries} tentatives")
    return False

prompt_test = (
    f"Génère du code Plotly Express. DataFrame 'df' déjà chargé.\n"
    f"Colonnes : {list(df.columns)}\n"
    "Demande : Top 10 des industries par nombre d'offres, bar chart horizontal trié.\n"
    "Termine avec fig.write_html('output.html'). Code dans ```python...```"
)

print("Lancement du repair loop...")
repair_loop(prompt_test, df)

In [ ]:
# ✏️ À COMPLÉTER — Étape 2C
import ollama, re, pandas as pd, plotly.express as px

df = pd.read_csv('job_postings.csv')

# TODO : Appelez repair_loop avec max_retries=5 sur une question plus complexe
# TODO : Question suggérée : "Comparaison des salaires min/max par type de poste, bar chart groupé"
# TODO : Notez combien de tentatives ont été nécessaires

# Votre code ici :

---
## Exercice 3 — Interface Streamlit

> **🎯 Objectif :** Wrapper le pipeline dans une interface web, d'abord minimale, puis connectée au pipeline LLM.

> **⏱️ Durée estimée : 35 minutes**

> **⚠️ Important :** Les cellules suivantes **écrivent des fichiers `.py`**. Lancez-les dans un terminal séparé avec `python -m streamlit run <fichier>.py`.

### Étape 3A — Application Streamlit minimale

Une app de 20 lignes : champ texte + bouton + affichage. Objectif : vérifier que Streamlit fonctionne.

In [ ]:
# NE PAS MODIFIER - Étape 3A : app Streamlit minimale (app_s8_minimal.py)
app_minimal = '''import streamlit as st

st.set_page_config(page_title="DataViz IA - Minimal", layout="centered")
st.title("DataViz IA — Test minimal")

# Champ texte : l'utilisateur tape sa question analytique
question = st.text_input("Votre question :", placeholder="Top 10 des industries...")

# Bouton : déclenche le traitement
if st.button("Générer", type="primary"):
    if question:
        # Pour l'instant on affiche juste la question — le LLM sera connecté à l'étape 3B
        st.write(f"Question reçue : **{question}**")
        st.info("Pipeline LLM non encore connecté — voir étape 3B")
    else:
        st.warning("Tapez une question avant de cliquer sur Générer")
'''

with open('app_s8_minimal.py', 'w', encoding='utf-8') as f:
    f.write(app_minimal)

print("✓ app_s8_minimal.py créé")
print("Lancez dans un terminal : python -m streamlit run app_s8_minimal.py")

In [ ]:
# ✏️ À COMPLÉTER — Étape 3A
# TODO : Modifiez app_minimal pour ajouter un graphique Plotly statique
#         (sans LLM, juste un px.bar codé en dur) affiché avec st.plotly_chart()
# TODO : Ajoutez un st.sidebar.header("Options") avec un selectbox de type de graphique

app_avec_chart = '''import streamlit as st
import plotly.express as px
import pandas as pd

st.set_page_config(page_title="DataViz IA - Avec chart", layout="wide")
st.title("DataViz IA — Avec graphique Plotly")

df = pd.read_csv("job_postings.csv")

# TODO : Ajoutez un champ texte et un bouton
# TODO : Affichez un graphique px.bar statique avec st.plotly_chart()
# TODO : Ajoutez dans la sidebar un selectbox pour choisir le type de chart

'''

# TODO : Écrivez app_avec_chart dans 'app_s8_avec_chart.py'
# with open('app_s8_avec_chart.py', 'w', encoding='utf-8') as f:
#     f.write(app_avec_chart)
# print("Lancez avec : python -m streamlit run app_s8_avec_chart.py")

### Étape 3B — Connecter le pipeline LLM à Streamlit

On intègre `repair_loop` dans l'interface : la question de l'utilisateur est envoyée au LLM, le graphique s'affiche.

In [ ]:
# NE PAS MODIFIER - Étape 3B : app Streamlit avec pipeline LLM (app_s8_bis.py)
app_pipeline = 'import streamlit as st\nimport pandas as pd, plotly.express as px, ollama, re\n\nst.set_page_config(page_title="DataViz IA", layout="wide")\nst.title("DataViz IA — Pipeline LLM")\n\ndf = pd.read_csv("job_postings.csv")\ncolonnes = list(df.columns)\n\ndef generer_graphique(question: str, max_retries: int = 3):\n    """Retourne (fig, nb_tentatives) ou (None, nb_tentatives) si echec."""\n    prompt = (\n        "Genere du code Plotly Express. DataFrame df disponible.\\n"\n        f"Colonnes : {colonnes}\\n"\n        f"Question : {question}\\n"\n        "Termine avec fig.show(). Code dans ```python...```"\n    )\n    messages = [{"role": "user", "content": prompt}]\n\n    for attempt in range(1, max_retries + 1):\n        resp = ollama.chat(model="llama3.2", messages=messages)\n        contenu = resp.message.content\n        match = re.search(r"```python\\n(.*?)```", contenu, re.DOTALL)\n        code = match.group(1).strip() if match else contenu.strip()\n        # On capture fig dans _fig au lieu de fig.show() pour l\'afficher dans Streamlit\n        code_capture = code.replace("fig.show()", "_fig = fig")\n        ctx = {"df": df, "px": px, "pd": pd}\n        try:\n            exec(code_capture, ctx)\n            if "_fig" in ctx:\n                return ctx["_fig"], attempt\n        except Exception as e:\n            if attempt < max_retries:\n                messages.append({"role": "assistant", "content": contenu})\n                messages.append({"role": "user", "content":\n                    f"Erreur : {e}\\nColonnes : {colonnes}\\nCorrige dans ```python...```"})\n    return None, max_retries\n\n# Interface principale\nquestion = st.text_input("Question analytique :", placeholder="Top 10 des industries...")\nif st.button("Generer le graphique", type="primary") and question:\n    with st.spinner("Pipeline en cours..."):\n        fig, nb = generer_graphique(question)\n    if fig:\n        st.success(f"Graphique genere en {nb} tentative(s)")\n        st.plotly_chart(fig, use_container_width=True)\n    else:\n        st.error(f"Echec apres {nb} tentatives — reformulez votre question")\n'

with open('app_s8_bis.py', 'w', encoding='utf-8') as f:
    f.write(app_pipeline)

print('✓ app_s8_bis.py créé')
print('Lancez dans un terminal : python -m streamlit run app_s8_bis.py')

In [ ]:
# ✏️ À COMPLÉTER — Étape 3B
# TODO : Modifiez app_s8_bis.py pour ajouter :
#   1. Un compteur st.metric("Tentatives", nb) sous le graphique
#   2. Un expander "Afficher le code généré" avec st.code(code)
#   3. En cas d'erreur : afficher le message dans st.error() avec st.expander

# Lisez app_s8_bis.py, modifiez-le, et relancez avec :
# python -m streamlit run app_s8_bis.py

# Votre code ici :

### Étape 3C — Lancer l'app depuis Jupyter

On peut lancer l'app directement depuis Jupyter avec `subprocess`.

In [ ]:
# NE PAS MODIFIER - Étape 3C : lancement de l'app
import subprocess

# python -m streamlit run : toujours utiliser cette forme sur ce système
subprocess.Popen(['python', '-m', 'streamlit', 'run', 'app_s8_bis.py'])
print("App lancée sur http://localhost:8501")
print("(Ouvrez votre navigateur si la page ne s'ouvre pas automatiquement)")

---
## Exercice 4 — Synthèse : pipeline complet en une cellule

> **🎯 Objectif :** Assembler les micro-étapes 1A→1F + 2C en un pipeline autonome.

> **⏱️ Durée estimée :** 20 minutes

> **📋 Consignes :**
> 1. Dans la cellule ci-dessous, assemblez TOUTES les étapes en une seule cellule cohérente
> 2. Testez avec **3 questions différentes** sur le dataset
> 3. Notez pour chaque question : nombre de tentatives, type d'erreur éventuelle, qualité du graphique

> **💡 Checklist :**
> - [ ] Étape 1A : chargement du CSV
> - [ ] Étape 1B : résumé numérique + catégoriel
> - [ ] Étape 1C : construction du prompt
> - [ ] Étape 1D : appel LLM
> - [ ] Étape 1E : extraction du code Python
> - [ ] Étape 2C : repair loop avec max_retries=3

In [ ]:
# ✅ SOLUTION — Exercice 4 : pipeline complet
import pandas as pd, plotly.express as px, plotly.io as pio, ollama, re, webbrowser, json

pio.renderers.default = 'browser'

# Étape 1A — chargement
df = pd.read_csv('job_postings.csv')

# Étape 1B — résumé dataset
numeric_stats = {}
for col in df.select_dtypes(include='number').columns:
    s = df[col].dropna()
    if len(s) > 0:
        numeric_stats[col] = {'min': round(float(s.min()), 2),
                               'max': round(float(s.max()), 2),
                               'mean': round(float(s.mean()), 2)}

categorical_tops = {}
for col in [c for c in df.columns if df[c].dtype == object]:
    categorical_tops[col] = df[col].value_counts().head(5).index.tolist()

# Étape 1E — extraction du code Python depuis la réponse LLM
def extract_python_code(text):
    match = re.search(r'```python\n(.*?)```', text, re.DOTALL)
    if match: return match.group(1).strip()
    match = re.search(r'```\n(.*?)```', text, re.DOTALL)
    if match: return match.group(1).strip()
    return text.strip()

def fix_code(code: str) -> str:
    # Supprime les imports matplotlib — le LLM les génère parfois par habitude
    lines = [l for l in code.splitlines()
             if not l.strip().startswith('import matplotlib')
             and not l.strip().startswith('from matplotlib')]
    code = '\n'.join(lines)
    # Corrige px.Bar → px.bar (capitalisation)
    code = re.sub(r'px\.([A-Z][a-z_]+)', lambda m: 'px.' + m.group(1).lower(), code)
    return code

# Étape 2C — repair loop avec correction renforcée à la tentative 3
def repair_loop(question, df, numeric_stats, categorical_tops, max_retries=3):
    cat_tops_str = json.dumps({k: v[:3] for k, v in categorical_tops.items()}, indent=2)
    prompt = f"""Code Python Plotly Express UNIQUEMENT dans ```python...```.
INTERDIT : matplotlib, seaborn. Utilise SEULEMENT plotly.express (px).
Question : {question}
Colonnes : {', '.join(df.columns.tolist())}
Stats numériques : {json.dumps(numeric_stats)[:300]}
Top catégoriels : {cat_tops_str[:300]}
DataFrame 'df' disponible. Termine avec fig.write_html('chart_ex4.html')."""

    context = {'df': df, 'px': px, 'pd': pd}
    messages = [{'role': 'user', 'content': prompt}]

    for attempt in range(1, max_retries + 1):
        print(f"  Tentative {attempt}/{max_retries}...")
        response = ollama.chat(model='llama3.2', messages=messages)
        code = extract_python_code(response.message.content)
        code = fix_code(code)
        code = code.replace('fig.show()', "fig.write_html('chart_ex4.html')")
        try:
            exec(code, context)
            print(f"  ✓ Succès à la tentative {attempt}")
            webbrowser.open('chart_ex4.html')
            return True
        except Exception as e:
            print(f"  ✗ Erreur : {e}")
            if attempt < max_retries:
                # Dès la tentative 3, rappel explicite : pas de matplotlib
                extra = ("\nRAPPEL : matplotlib est INTERDIT. Utilise UNIQUEMENT plotly.express (px)."
                         if attempt >= 2 else "")
                messages.append({'role': 'assistant', 'content': response.message.content})
                messages.append({'role': 'user', 'content':
                    f"Erreur : {e}\nColonnes : {list(df.columns)}{extra}\n"
                    f"Corrige dans ```python...```"})
    print(f"  ✗ Échec après {max_retries} tentatives")
    return False

questions_test = [
    "Top 10 des industries avec le plus d'offres, bar chart horizontal",
    "Distribution des salaires par type de poste, box plot",
    "Évolution du nombre d'offres par mois, line chart",
]

for q in questions_test:
    print(f"\nQuestion : {q}")
    succes = repair_loop(q, df, numeric_stats, categorical_tops)
    print(f"Résultat : {'✓ Succès' if succes else '✗ Échec'}")


---
## Synthèse — Ce que vous avez construit

### Le pipeline en 4 appels

```python
# Résumé : le pipeline complet en 4 lignes
numeric_stats, categorical_tops = resumer_dataset(df)
prompt  = construire_prompt(question, df, numeric_stats, categorical_tops)
code    = extract_python_code(ollama.chat(...).message.content)
succes  = repair_loop(code, df, max_retries=3)
```

### Points clés à retenir

| # | Concept | À retenir |
|---|---------|-----------|
| 1 | Résumé dataset | Jamais envoyer le CSV brut — toujours un résumé compact |
| 2 | Noms de colonnes exacts | Les inclure dans le prompt évite les hallucinations |
| 3 | `response.message.content` | API Ollama — pas `response['message']['content']` |
| 4 | Repair loop | 3 tentatives max, historique multi-tour, fallback explicite |
| 5 | Sandboxing | `exec(code, {'df': df, 'px': px, 'pd': pd})` uniquement |
| 6 | Streamlit | `python -m streamlit run` — pas `streamlit run` sur ce système |

### Transition vers la séance 9
Le pipeline est **statique** : il suit toujours les mêmes étapes. En séance 9, vous le rendrez **agentique** : l'agent décidera lui-même quelle étape exécuter, dans quel ordre, avec quels outils.